# Payphone Storytelling Model – Colab QLoRA Training

Train the Payphone legal fiction model on **Google Colab's free T4 GPU**.
Then download the LoRA adapter and use it locally with Ollama.

**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Install dependencies

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
!pip install -q unsloth bitsandbytes transformers datasets trl accelerate peft

## 2. Upload training data

Upload `training_payphone.jsonl` from your project (after running validate and convert scripts).

**Alternative:** Mount Google Drive and set `DATA_PATH` to your file.

In [ ]:
from google.colab import files
import os

print("Click 'Choose Files' and select training_payphone.jsonl")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0] if uploaded else None

# Option B: drive.mount('/content/drive'); DATA_PATH = '/content/drive/MyDrive/training_payphone.jsonl'

if DATA_PATH and os.path.exists(DATA_PATH):
    print(f"Using data: {DATA_PATH}")
else:
    raise FileNotFoundError("Upload training_payphone.jsonl or set DATA_PATH")

## 3. Load model and train

In [ ]:
import json
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MODEL = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "/content/payphone-storyteller-lora"
MAX_SEQ_LENGTH = 512   # T4 15GB: 1024 OOMs; 512 fits (some story truncation)
EPOCHS = 3
BATCH_SIZE = 1

def load_jsonl(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

records = load_jsonl(DATA_PATH)
print(f"Loaded {len(records)} examples")

In [ ]:
print("Loading model (4-bit QLoRA)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=4,  # r=8 OOMs; attention-only + r=4 fits T4
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded")

In [ ]:
import os

def format_chat(record, tokenizer):
    messages = record.get("messages", [])
    if not messages:
        return ""
    try:
        out = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        return out if isinstance(out, str) else ""
    except Exception as e:
        print(f"Warning: format_chat failed: {e}")
        return ""

texts = [format_chat(r, tokenizer) for r in records]
texts = [t for t in texts if t and t.strip()]
if not texts:
    raise ValueError("No valid training examples after formatting")

# Pre-truncate to avoid Unsloth cross_entropy shape mismatch
def truncate_to_tokens(text, tokenizer, max_len):
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_len:
        return text
    return tokenizer.decode(ids[:max_len], skip_special_tokens=False)

texts = [truncate_to_tokens(t, tokenizer, MAX_SEQ_LENGTH - 8) for t in texts]
dropped = 0
print(f"Formatted {len(texts)} examples" + (f" (dropped {dropped} invalid)" if dropped else ""))
dataset = Dataset.from_dict({"text": texts})
print(f"Dataset: {len(dataset)} rows")
print("\nSample (first 400 chars):")
print(dataset["text"][0][:400] + "..." if len(dataset["text"][0]) > 400 else dataset["text"][0])

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    fp16=not __import__("torch").cuda.is_bf16_supported(),
    bf16=__import__("torch").cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=15,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    packing=False,
)

print("Training... (~2-4 hours on T4)")
try:
    trainer.train()
except Exception as e:
    print(f"TRAINING FAILED: {e}")
    raise

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Verify files were saved before zipping
saved_files = os.listdir(OUTPUT_DIR) if os.path.exists(OUTPUT_DIR) else []
if not saved_files:
    raise RuntimeError(
        "Save produced no files! Check for errors above. "
        "Training may have failed silently or OUTPUT_DIR is wrong."
    )
print(f"Saved: {saved_files}")

## 4. Zip and download

In [ ]:
!cd /content && zip -r payphone-storyteller-lora.zip payphone-storyteller-lora

from google.colab import files
files.download("/content/payphone-storyteller-lora.zip")

print("Download started. See TRAINING_COLAB.md for Ollama import steps.")